# EURUSD_Data_Retrieval_and_Preparation.ipynb

## Retrieving and Preparing EUR/USD Data for Strategy Analysis

The goal of this notebook is to load historical EUR/USD exchange rate data from a CSV file, inspect the dataset, visualize the price series, and prepare daily log returns for later strategy testing and backtesting.

This is a foundational step in any data-driven trading workflow: before testing a strategy, we need clean, correctly indexed price data and a reliable return series.

## 1. Import Libraries

We use a small Python stack:

- `pandas` for loading and working with time series data
- `numpy` for calculating log returns
- `matplotlib` for visualization

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Keep numeric output readable.
pd.options.display.float_format = '{:,.6f}'.format

# Use a clean default matplotlib style.
plt.style.use('default')

## 2. Load the EUR/USD CSV File

The expected file is `EURUSD.csv` in the same folder as this notebook. The file should contain:

- one date column
- one EUR/USD price column

If the CSV is not available, this notebook creates a reproducible synthetic EUR/USD-style dataset so the lesson still runs. When real data is available, the CSV is used instead.

In [ ]:
def create_synthetic_eurusd_data(seed=42):
    """Create reproducible daily EUR/USD-like data when EURUSD.csv is unavailable."""
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range(start='2018-01-02', end='2024-12-31')

    # FX daily changes are usually small, so we use modest drift and volatility.
    daily_log_returns = rng.normal(loc=0.00001, scale=0.0045, size=len(dates))
    price = 1.18 * np.exp(np.cumsum(daily_log_returns))

    data = pd.DataFrame({'EURUSD': price}, index=dates)
    data.index.name = 'Date'
    return data, 'Synthetic EUR/USD data'


def find_datetime_column(dataframe):
    """Find a likely date column using common names first, then parsability."""
    common_date_columns = ['Date', 'Datetime', 'DateTime', 'Timestamp', 'time', 'Time']

    for column in common_date_columns:
        if column in dataframe.columns:
            return column

    for column in dataframe.columns:
        if pd.api.types.is_numeric_dtype(dataframe[column]):
            continue

        parsed_dates = pd.to_datetime(dataframe[column], errors='coerce')
        if parsed_dates.notna().mean() > 0.80:
            return column

    raise ValueError('Could not identify a date column. Please inspect the CSV columns manually.')


def load_eurusd_csv(filename='EURUSD.csv'):
    """Load EUR/USD data from CSV and return a clean time-indexed DataFrame."""
    csv_path = Path(filename)

    if not csv_path.exists():
        return create_synthetic_eurusd_data()

    raw_data = pd.read_csv(csv_path)
    date_column = find_datetime_column(raw_data)

    raw_data[date_column] = pd.to_datetime(raw_data[date_column], errors='coerce')
    data = raw_data.dropna(subset=[date_column]).set_index(date_column).sort_index()
    data.index.name = 'Date'

    # Keep numeric price columns only. The expected file has one price column.
    for column in data.columns:
        data[column] = pd.to_numeric(data[column], errors='coerce')

    price_columns = data.select_dtypes(include='number').columns.tolist()

    if len(price_columns) == 0:
        raise ValueError('No numeric price column found in EURUSD.csv.')

    price_column = price_columns[0]
    data = data[[price_column]].rename(columns={price_column: 'EURUSD'})

    return data.dropna(), f'Loaded from {csv_path}'


eurusd_data, data_source = load_eurusd_csv('EURUSD.csv')

print(f'Data source: {data_source}')
eurusd_data.head()

## 3. Inspect the Dataset

Before calculating returns or testing strategies, we should inspect the data. This helps confirm that dates were parsed correctly, the index is chronological, and the price column is numeric.

In [ ]:
# First few rows.
eurusd_data.head()

In [ ]:
# Dataset structure and data types.
eurusd_data.info()

In [ ]:
# Summary statistics for the EUR/USD price column.
eurusd_data.describe()

## 4. Date Range and Observations

For time series data, the date range and number of observations matter. They tell us how much history we have available for strategy analysis.

EUR/USD data is usually trading-day data. Weekends and some non-trading days may be missing because foreign exchange markets do not trade continuously in the same way across every calendar day.

In [ ]:
start_date = eurusd_data.index.min()
end_date = eurusd_data.index.max()
number_of_observations = len(eurusd_data)

print(f'Start date: {start_date.date()}')
print(f'End date: {end_date.date()}')
print(f'Number of observations: {number_of_observations:,}')
print(f'Index type: {type(eurusd_data.index)}')

## 5. Visualize the EUR/USD Price Series

A simple price chart is often the fastest way to understand the broad behavior of the dataset. We can see trends, ranges, and large moves before moving into return calculations.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(eurusd_data.index, eurusd_data['EURUSD'], label='EUR/USD')
plt.title('EUR/USD Exchange Rate')
plt.xlabel('Date')
plt.ylabel('Exchange Rate')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Calculate Daily Log Returns

Strategy testing usually works with returns rather than raw prices.

Here we calculate daily log returns with:

`returns = log(price_t / price_t-1)`

This compares today's EUR/USD price with the previous available price.

In [ ]:
# Calculate daily log returns from the EUR/USD price column.
eurusd_data['returns'] = np.log(eurusd_data['EURUSD'] / eurusd_data['EURUSD'].shift(1))

eurusd_data.head()

The first return is missing because there is no previous price to compare with on the first row.

A log return represents the continuously compounded return between two prices. Log returns are often preferred in quantitative finance and backtesting because they are time-additive, easier to aggregate across periods, and behave well in many statistical workflows.

## 7. Check for Missing Values

Before using this data in a strategy, we should check for missing values. Missing price values or returns can break calculations or distort a backtest.

In [ ]:
missing_values = eurusd_data.isna().sum()
print(missing_values)

print('\nComment:')
if missing_values['returns'] == 1 and missing_values['EURUSD'] == 0:
    print('Only the first return is missing, which is expected after using shift(1).')
else:
    print('There are additional missing values. Inspect and clean them before strategy testing.')

## 8. Clean Return Series for Backtesting

For most backtests, we remove the first row with the missing return so the strategy has a clean return series to work with.

In [ ]:
eurusd_clean = eurusd_data.dropna().copy()

print(f'Rows before dropping missing returns: {len(eurusd_data):,}')
print(f'Rows after dropping missing returns:  {len(eurusd_clean):,}')

eurusd_clean.head()

## Key Takeaways

- We loaded EUR/USD exchange rate data from `EURUSD.csv`, or used a synthetic fallback if the file was unavailable.
- We parsed dates and used a `DatetimeIndex`, which is essential for financial time series analysis.
- We inspected the dataset with `.head()`, `.info()`, and `.describe()` before doing calculations.
- We visualized the EUR/USD price series to understand the broad behavior of the data.
- We calculated daily log returns with `np.log(price / price.shift(1))`.
- The first return is missing because there is no previous price for comparison.
- This preparation matters because strategy testing and backtesting require clean, correctly indexed price and return data.